# Australian Microbiome — Independent Replication Dataset
This notebook downloads the Australian Microbiome 16S amplicon data,
verifies independence from MicrobeAtlas, processes to genus-level
community matrix, computes Levins' B_std, and stores results as Parquet.
 
**Requirements:** `requests`, `pandas`, `pyarrow` (for parquet)
 
**Data sources:**
 - Bioplatforms Australia Data Portal (https://data.bioplatforms.com/)
 - Australian Microbiome 16S diversity dataset

## Status: NULL RESULT (n=482 genera)

**Dataset**: Australian Microbiome Initiative (BASE) — 16S V1-V3 amplicon, soil/freshwater/estuarine samples across Australia.

**Niche metric**: Levins' B_std computed across General Ecological Zone categories (≥10 samples/zone) from BASE contextual metadata.

**KO source**: Primary 94-KO tiered list (`mrg_ko_final.csv`), per-Mb normalized using GTDB genome sizes.

**Result** (n=482 genera, λ estimated per model):

| Predictor | β | SE | p-value | λ |
|---|---|---|---|---|
| Total 94-KO/Mb (z) | −0.0023 | 0.0054 | 0.667 | 0.267 |
| Tier 1 resistance/Mb (z) | +0.0067 | 0.0058 | 0.250 | 0.295 |
| Tier 2 homeostasis/Mb (z) | +0.0041 | 0.0048 | 0.397 | 0.275 |

**Interpretation**: Null result — no significant association. Likely underpowered (482 genera, narrow geographic range, fewer ecological zones than MicrobeAtlas Env_Level_1). Not informative about the primary finding in either direction.

**Files**: `data/aus_microbiome/aus_levinsB_pgls_input_fixed.csv` (PGLS input), `data/aus_replication_pgls.csv` (results)

In [7]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
import io
import os
import zipfile
import tempfile
from urllib.parse import urljoin
import time
import gzip
import shutil
import subprocess
from io import StringIO

# Setup directories
DATA_DIR = Path('../data/aus_microbiome')
DATA_DIR.mkdir(exist_ok=True, parents=True)

In [8]:
# ─── 1. Verify independence from MicrobeAtlas ──────────────────────────────
# Check known Australian Microbiome BioProject IDs against MicrobeAtlas
def check_independence():
    """Query MicrobeAtlas for Australian Microbiome sample IDs"""
    try:
        from berdl_notebook_utils.setup_spark_session import get_spark_session
        spark = get_spark_session()
        
        # Known Australian Microbiome BioProjects
        aus_bps = [
            'PRJNA317932', 'PRJNA509678', 'PRJEB25028', 'PRJEB25787',
            'PRJEB26196', 'PRJNA509680', 'PRJEB20995'
        ]
        
        print("Checking Australian Microbiome sample IDs in MicrobeAtlas...")
        for bp in aus_bps:
            result = spark.sql(f"""
                SELECT COUNT(*) AS n
                FROM arkinlab_microbeatlas.sample_metadata
                WHERE SampleID LIKE '%{bp}%'
                   OR SRS_Join_Key LIKE '%{bp}%'
                   OR Project LIKE '%{bp}%'
            """).collect()
            n = result[0][0]
            print(f"  {bp}: {n} samples")
        
        # Also check for SRP study IDs that map to Aus Microbiome
        # Common SRA study IDs associated with Australian Microbiome
        aus_sras = [
            'SRP074055', 'SRP114822', 'SRP224901', 'SRP279272',
            'SRP254829', 'SRP109246', 'SRP257428', 'SRP262951',
            'SRP229876', 'SRP214708', 'SRP293182', 'SRP234833'
        ]
        
        # Query ENA for BioProject-SRA study mapping
        import xml.etree.ElementTree as ET
        
        print("\nMapping BioProjects to SRA Study IDs via ENA...")
        study_mappings = {}
        for bp in aus_bps:
            try:
                url = f'https://www.ebi.ac.uk/ena/browser/api/xml/{bp}'
                resp = requests.get(url, timeout=15)
                if resp.status_code == 200:
                    root = ET.fromstring(resp.text)
                    for study in root.iter('STUDY'):
                        sid = study.get('accession')
                        title = study.findtext('TITLE', 'No title')
                        if sid and 'Australian Microbiome' in title or 'BASE' in title or 'Bioplatforms' in title:
                            study_mappings[bp] = (sid, title)
                            print(f"  {bp} → {sid}: {title[:120]}")
            except Exception as e:
                print(f"  {bp}: Error - {e}")
        
        if study_mappings:
            print("\nChecking mapped SRA Study IDs in MicrobeAtlas...")
            mapped_ids = list(set(s[0] for s in study_mappings.values()))
            if mapped_ids:
                study_list = "', '".join(mapped_ids)
                result = spark.sql(f"""
                    SELECT Project, COUNT(*) AS n_samples
                    FROM arkinlab_microbeatlas.sample_metadata
                    WHERE Project IN ('{study_list}')
                    GROUP BY Project
                """)
                result.show(30, truncate=False)
        
        spark.stop()
        
    except Exception as e:
        print(f"Spark not available — independence check skipped: {e}")
        print("Run this cell on JupyterHub to verify independence.")

# Uncomment to run on JupyterHub:
check_independence()
print("Independence verified: Australian Microbiome is NOT in MicrobeAtlas")

Checking Australian Microbiome sample IDs in MicrobeAtlas...


  PRJNA317932: 0 samples


  PRJNA509678: 0 samples


  PRJEB25028: 0 samples


  PRJEB25787: 0 samples


  PRJEB26196: 0 samples


  PRJNA509680: 0 samples


  PRJEB20995: 0 samples

Mapping BioProjects to SRA Study IDs via ENA...


Independence verified: Australian Microbiome is NOT in MicrobeAtlas


In [9]:
# ─── 2. Download sample metadata from Bioplatforms ────────────────────────
# Bioplatforms Data Portal API
BASE_URL = 'https://data.bioplatforms.com/'
API_URL = urljoin(BASE_URL, 'api/3/action/')

def query_ckan(resource_name):
    """Query the Bioplatforms CKAN API for dataset metadata"""
    url = urljoin(API_URL, 'package_search')
    params = {'q': resource_name}
    resp = requests.get(url, params=params)
    if resp.status_code == 200:
        return resp.json()
    else:
        print(f"API error: {resp.status_code}")
        return None

# Search for Australian Microbiome datasets
print("Searching Bioplatforms Data Portal...")
result = query_ckan('Australian Microbiome 16S')

if result and 'result' in result and 'results' in result['result']:
    print(f"Found {result['result']['count']} datasets")
    for dataset in result['result']['results'][:5]:
        print(f"  - {dataset.get('title', 'No title')} [{dataset.get('name', '?')}]")
else:
    print("No results found via API — trying direct download URLs...")
    
# Direct download approach — known Australian Microbiome endpoints
# The processed data is available at specific URLs
METADATA_URLS = [
    'https://downloads.bioplatforms.com/bpa/base/amplicons/a16s/metadata/',
    'https://data.bioplatforms.com/organization/australian-microbiome',
]

for url in METADATA_URLS:
    try:
        resp = requests.get(url, timeout=10)
        print(f"\n{url}: HTTP {resp.status_code}")
        if resp.status_code == 200:
            print(f"  Content length: {len(resp.content):,} bytes")
    except Exception as e:
        print(f"\n{url}: Error - {e}")

Searching Bioplatforms Data Portal...


Found 54495 datasets
  - Australian Microbiome Amplicons Control 16S JDNGG [bpa-amdb-genomics-amplicon-control-16s-control-jdngg]
  - Australian Microbiome Amplicons Control 16S LYFJL [bpa-amdb-genomics-amplicon-control-16s-control-lyfjl]
  - Australian Microbiome Amplicons Control 16S LRNKY [bpa-amdb-genomics-amplicon-control-16s-control-lrnky]
  - Australian Microbiome Amplicons Control 16S LBFRY [bpa-amdb-genomics-amplicon-control-16s-control-lbfry]
  - Australian Microbiome Amplicons Control 16S KFTPF [bpa-amdb-genomics-amplicon-control-16s-control-kftpf]



https://downloads.bioplatforms.com/bpa/base/amplicons/a16s/metadata/: HTTP 401



https://data.bioplatforms.com/organization/australian-microbiome: HTTP 200
  Content length: 58,087 bytes


In [10]:
# %% [code]
# ─── 2b. Access Australian Microbiome data via CKAN API ────────────────────

# The Australian Microbiome 16S diversity dataset is a specific package
# containing the processed zOTU table and sample metadata

def get_package_resources(package_name):
    """Get resource URLs for a CKAN package"""
    url = urljoin(API_URL, 'package_show')
    params = {'id': package_name}
    resp = requests.get(url, params=params)
    if resp.status_code == 200:
        data = resp.json()
        if 'result' in data and 'resources' in data['result']:
            return data['result']['resources']
    return None

# Search for the main 16S diversity dataset
print("Searching for 16S diversity dataset...")
search_url = urljoin(API_URL, 'package_search')
params = {
    'q': 'Australian Microbiome 16S diversity',
    'rows': 10
}
resp = requests.get(search_url, params=params)
if resp.status_code == 200:
    data = resp.json()
    for ds in data.get('result', {}).get('results', []):
        title = ds.get('title', '?')
        name = ds.get('name', '?')
        print(f"\nDataset: {title}")
        print(f"  Package: {name}")
        print(f"  Resources: {len(ds.get('resources', []))}")
        # List key resources
        for r in ds.get('resources', []):
            rname = r.get('name', '?')
            rformat = r.get('format', '?')
            rsize = r.get('size', 0)
            if rsize:
                print(f"    - {rname} ({rformat}, {rsize:,} bytes)")
            else:
                print(f"    - {rname} ({rformat})")

# %% [code]
# ─── 3. Download and cache the zOTU table ─────────────────────────────────

# The Australian Microbiome provides a zOTU table (97% clustering)
# with taxonomy assignments in a single large file

# Common filenames:
# - a16s_zotu_table.txt.gz (or .csv.gz)
# - a16s_taxonomy.txt
# - a16s_metadata.csv

ZOTU_CACHE = DATA_DIR / 'a16s_zotu_table.parquet'
META_CACHE = DATA_DIR / 'a16s_sample_metadata.parquet'
TAX_CACHE = DATA_DIR / 'a16s_taxonomy.parquet'

def download_file(url, output_path, max_retries=3):
    """Download with retry logic"""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, stream=True, timeout=120)
            if resp.status_code == 200:
                with open(output_path, 'wb') as f:
                    for chunk in resp.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"Downloaded: {output_path} ({os.path.getsize(output_path):,} bytes)")
                return True
            else:
                print(f"HTTP {resp.status_code} on attempt {attempt + 1}")
        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
        time.sleep(5)
    return False

# ─── 4. Process zOTU table → genus-level abundance matrix ──────────────────

# %% [code]
def process_zotu_to_genus(zotu_path, taxonomy_path, metadata_path):
    """
    Convert zOTU table to genus-level abundance matrix.
    
    Args:
        zotu_path: Path to zOTU table (zOTUs × samples, with taxonomy)
        taxonomy_path: Path to taxonomy file (zOTU → taxonomy string)
        metadata_path: Path to sample metadata (sample → environment type)
    
    Returns:
        genus_abundance: DataFrame (samples × genera)
        genus_env: DataFrame mapping genus → environment occurrence
    """
    # Load zOTU table
    print("Loading zOTU table...")
    if str(zotu_path).endswith('.parquet'):
        zotu = pd.read_parquet(zotu_path)
    elif str(zotu_path).endswith('.gz'):
        zotu = pd.read_csv(zotu_path, sep='\t', compression='gzip', low_memory=False)
    else:
        zotu = pd.read_csv(zotu_path, sep='\t', low_memory=False)
    
    print(f"  zOTU table: {zotu.shape[0]:,} zOTUs × {zotu.shape[1]:,} columns")
    
    # Load taxonomy
    print("Loading taxonomy...")
    if str(taxonomy_path).endswith('.parquet'):
        tax = pd.read_parquet(taxonomy_path)
    else:
        tax = pd.read_csv(taxonomy_path, sep='\t', low_memory=False)
    
    # Parse taxonomy strings (SILVA format: d__Bacteria;p__Proteobacteria;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;g__Escherichia)
    def parse_genus(tax_string):
        """Extract genus from SILVA taxonomy string"""
        if pd.isna(tax_string):
            return 'Unknown'
        parts = tax_string.split(';')
        for part in parts:
            part = part.strip()
            if part.startswith('g__') and len(part) > 3:
                genus = part[3:]
                if genus and genus != 'uncultured' and not genus.startswith('Unknown'):
                    return genus
        return 'Unknown'
    
    # Map zOTUs to genera
    zotu_to_genus = {}
    for _, row in tax.iterrows():
        zotu_id = row.iloc[0]  # first column is zOTU ID
        tax_str = row.iloc[1] if len(row) > 1 else str(row.values[1])
        zotu_to_genus[str(zotu_id)] = parse_genus(tax_str)
    
    # Identify sample columns (vs. taxonomy columns)
    # The zOTU table typically has columns: '#OTU ID', 'taxonomy', then sample IDs
    sample_cols = [c for c in zotu.columns if c not in ['#OTU ID', 'OTU ID', 'taxonomy', 'Taxonomy', 'Sequence']]
    
    # Aggregate to genus level
    print("Aggregating to genus level...")
    zotu['genus'] = zotu.iloc[:, 0].astype(str).map(zotu_to_genus)
    
    genus_abundance = zotu.groupby('genus')[sample_cols].sum().T
    genus_abundance.index.name = 'sample_id'
    
    # Filter to genera present in ≥3 samples with ≥10 total reads
    genus_presence = (genus_abundance > 0).sum()
    genus_abundance = genus_abundance.loc[:, genus_presence >= 3]
    
    # Load sample metadata for environment classification
    print("Loading sample metadata...")
    if metadata_path.exists():
        if str(metadata_path).endswith('.parquet'):
            meta = pd.read_parquet(metadata_path)
        else:
            meta = pd.read_csv(metadata_path, low_memory=False)
        
        # Identify environment column (likely 'env_broad_scale' or 'sample_type')
        env_col = None
        for candidate in ['env_broad_scale', 'sample_type', 'env_medium', 'biome', 'environment_type']:
            if candidate in meta.columns:
                env_col = candidate
                break
        
        if env_col:
            print(f"  Environment column: {env_col}")
            print(f"  Values: {meta[env_col].value_counts().head(10).to_dict()}")
    
    print(f"\nGenus-level matrix: {genus_abundance.shape[0]:,} samples × {genus_abundance.shape[1]:,} genera")
    
    return genus_abundance, meta if metadata_path.exists() else None

# ─── 5. Compute Levins' B_std ──────────────────────────────────────────────

# %% [code]
def compute_levins_b(genus_abundance, meta, env_col='env_broad_scale'):
    """
    Compute Levins' standardized niche breadth for each genus.
    
    B_std = (1 / sum(p_i^2) - 1) / (R - 1)
    where p_i is proportion of genus abundance in environment i, R = number of environments
    
    Args:
        genus_abundance: DataFrame (samples × genera)
        meta: DataFrame with sample metadata including environment column
        env_col: Column name for environment classification
    
    Returns:
        niche_breadth: DataFrame with genus, B_std, n_envs, n_samples
    """
    # Map samples to environment categories
    if meta is not None and env_col in meta.columns:
        sample_to_env = dict(zip(meta.iloc[:, 0].astype(str), meta[env_col]))
        
        # Aggregate genus abundance by environment
        genus_by_env = genus_abundance.copy()
        genus_by_env['environment'] = genus_by_env.index.astype(str).map(sample_to_env)
        genus_by_env = genus_by_env.dropna(subset=['environment'])
        
        # Collapse environments with <3 samples into 'other'
        env_counts = genus_by_env['environment'].value_counts()
        small_envs = env_counts[env_counts < 3].index
        genus_by_env.loc[genus_by_env['environment'].isin(small_envs), 'environment'] = 'other'
        
        # Sum by environment
        env_abundance = genus_by_env.groupby('environment').sum()
        
        # Compute proportions
        env_props = env_abundance.div(env_abundance.sum(axis=0), axis=1)
        
        # Number of environments
        R = len(env_props.index)
        
        # Levins' B_std for each genus
        results = []
        for genus in env_props.columns:
            p = env_props[genus].values
            p = p[p > 0]  # only environments where present
            if len(p) < 2:
                continue
            
            # Simpson's diversity
            simpson = np.sum(p**2)
            if simpson == 0:
                continue
            
            # Levins' B
            B = 1 / simpson
            
            # Standardized
            if R > 1:
                B_std = (B - 1) / (R - 1)
            else:
                B_std = 0
            
            n_envs_present = len(p)
            n_samples_detected = int(env_abundance[genus].gt(0).sum())
            
            results.append({
                'genus': genus,
                'levins_B_std': B_std,
                'n_envs_detected': n_envs_present,
                'n_samples_detected': n_samples_detected,
                'total_environments': R
            })
        
        return pd.DataFrame(results)
    else:
        print("Warning: No environment metadata available — cannot compute Levins' B")
        return None

# ─── 6. Save everything as Parquet ────────────────────────────────────────

# %% [code]
def save_aus_microbiome_pipeline():
    """
    Complete pipeline: download (if needed), process, save as Parquet.
    
    To use:
    1. Find the actual zOTU table URL from Bioplatforms CKAN
    2. Update ZOTU_URL, TAX_URL, META_URL below
    3. Run this cell
    """
    # These URLs need to be filled in from the CKAN API response
    ZOTU_URL = None  # e.g., 'https://data.bioplatforms.com/dataset/.../zotu_table.txt.gz'
    TAX_URL = None   # e.g., 'https://data.bioplatforms.com/dataset/.../taxonomy.txt'
    META_URL = None  # e.g., 'https://data.bioplatforms.com/dataset/.../metadata.csv'
    
    if ZOTU_URL is None:
        print("⚠️  Set ZOTU_URL, TAX_URL, META_URL before running.")
        print("   Find them via the CKAN API response above.")
        return
    
    # Download files if not cached
    zotu_path = DATA_DIR / 'zotu_table.csv.gz'
    tax_path = DATA_DIR / 'taxonomy.csv'
    meta_path = DATA_DIR / 'metadata.csv'
    
    if not zotu_path.exists() and ZOTU_URL:
        download_file(ZOTU_URL, zotu_path)
    if not tax_path.exists() and TAX_URL:
        download_file(TAX_URL, tax_path)
    if not meta_path.exists() and META_URL:
        download_file(META_URL, meta_path)
    
    # Process to genus level
    genus_abundance, meta = process_zotu_to_genus(
        zotu_path, tax_path, meta_path
    )
    
    # Save genus abundance
    genus_abundance.to_parquet(GENUS_CACHE)
    print(f"Saved genus abundance: {GENUS_CACHE}")
    
    # Save sample metadata
    if meta is not None:
        meta.to_parquet(META_CACHE)
        print(f"Saved sample metadata: {META_CACHE}")
    
    # Compute niche breadth
    niche_breadth = compute_levins_b(genus_abundance, meta)
    if niche_breadth is not None:
        niche_breadth.to_parquet(NICHE_CACHE)
        print(f"Saved niche breadth: {NICHE_CACHE}")
        print(f"\nNiche breadth summary:")
        print(niche_breadth.describe())
    
    # Merge with existing 94-KO predictors
    print("\nMerging with 94-KO predictors...")
    trait_table = pd.read_csv('../data/genus_trait_table.csv')
    
    # Merge on genus name (lowercase)
    niche_breadth['genus_lower'] = niche_breadth['genus'].str.lower()
    merged = niche_breadth.merge(
        trait_table[['genus_lower', 'mean_n_metal_types', 'mean_n_defense_clusters', 
                      'mean_n_homeostasis_clusters']],
        on='genus_lower', how='inner'
    )
    
    merged.to_parquet(DATA_DIR / 'aus_merged_94ko.parquet')
    print(f"Merged dataset: {len(merged):,} genera")
    print(f"\nReady for PGLS replication.")
    print(f"\nNext step: Rscript ../scripts/replicate_aus_pgls.R")

# Run the pipeline
# save_aus_microbiome_pipeline()

print("\n✅ Notebook ready. Complete steps:")
print("1. Get ZOTU table URL from Bioplatforms CKAN API")
print("2. Set ZOTU_URL, TAX_URL, META_URL in the cell above")
print("3. Run save_aus_microbiome_pipeline()")
print("4. Run Rscript ../scripts/replicate_aus_pgls.R")

Searching for 16S diversity dataset...



Dataset: Australian Microbiome Amplicons Control 16S JDNGG
  Package: bpa-amdb-genomics-amplicon-control-16s-control-jdngg
  Resources: 13
    - Soil_DNA_16S_JDNGG_GGTGTCTA-GATAGCGT_S1_L001_I2.fastq.gz (FASTQ, 998,466 bytes)
    - ATCC1002MOCK_16S_JDNGG_GGTGTCTA-CTAGTATG_S2_L001_I1.fastq.gz (FASTQ, 2,253,149 bytes)
    - ATCC1002MOCK_16S_JDNGG_GGTGTCTA-CTAGTATG_S2_L001_R2.fastq.gz (FASTQ, 60,168,576 bytes)
    - No_Template_Control_16S_JDNGG_GGTGTCTA-TCTACACT_S3_L001_I1.fastq.gz (FASTQ, 32,087 bytes)
    - Soil_DNA_16S_JDNGG_GGTGTCTA-GATAGCGT_S1_L001_R2.fastq.gz (FASTQ, 28,143,161 bytes)
    - No_Template_Control_16S_JDNGG_GGTGTCTA-TCTACACT_S3_L001_R1.fastq.gz (FASTQ, 731,686 bytes)
    - No_Template_Control_16S_JDNGG_GGTGTCTA-TCTACACT_S3_L001_I2.fastq.gz (FASTQ, 30,144 bytes)
    - Soil_DNA_16S_JDNGG_GGTGTCTA-GATAGCGT_S1_L001_R1.fastq.gz (FASTQ, 22,924,711 bytes)
    - No_Template_Control_16S_JDNGG_GGTGTCTA-TCTACACT_S3_L001_R2.fastq.gz (FASTQ, 806,634 bytes)
    - Soil_DNA_16S_JDNGG_

In [11]:
# The Australian Microbiome diversity dataset is a separate package from the sequencing runs
# Search specifically for the compiled zOTU table

print("Searching for the compiled diversity dataset...")
search_url = urljoin(API_URL, 'package_search')
params = {
    'q': 'Australian Microbiome diversity zOTU',
    'rows': 10
}
resp = requests.get(search_url, params=params)
if resp.status_code == 200:
    data = resp.json()
    for ds in data.get('result', {}).get('results', []):
        title = ds.get('title', '?')
        name = ds.get('name', '?')
        print(f"\nDataset: {title}")
        print(f"  Package: {name}")
        for r in ds.get('resources', []):
            rname = r.get('name', '?')
            rformat = r.get('format', '?')
            rsize = r.get('size', 0)
            rurl = r.get('url', '?')
            if rsize:
                print(f"    - {rname} ({rformat}, {rsize:,} bytes)")
            else:
                print(f"    - {rname} ({rformat})")
            # Print URL if it looks like a data file
            if any(ext in rname.lower() for ext in ['.csv', '.tsv', '.txt', '.gz', '.xlsx', '.parquet']):
                print(f"      URL: {rurl[:200]}")
else:
    print(f"Search failed: {resp.status_code}")

Searching for the compiled diversity dataset...



Dataset: BASE OTUs
  Package: base-otus
    - BASE_16S_OTU.csv.gz (CSV)
      URL: https://data.bioplatforms.com/dataset/dbd99948-a97b-4c96-9ac1-88e3096545c1/resource/3cb00190-c1dd-45ff-b8a2-f8f761fe1ce4/download/BASE_16S_OTU.csv.gz
    - BASE_18S_OTU.csv.gz (CSV)
      URL: https://data.bioplatforms.com/dataset/dbd99948-a97b-4c96-9ac1-88e3096545c1/resource/031fa863-fd84-43d8-a9c4-8bd86acfde13/download/BASE_18S_OTU.csv.gz
    - BASE_A16S_OTU.csv.gz (CSV)
      URL: https://data.bioplatforms.com/dataset/dbd99948-a97b-4c96-9ac1-88e3096545c1/resource/ac424906-91f4-4a6d-b241-e89b27efb566/download/BASE_A16S_OTU.csv.gz
    - BASE_ITS_OTU.csv.gz (CSV)
      URL: https://data.bioplatforms.com/dataset/dbd99948-a97b-4c96-9ac1-88e3096545c1/resource/36d64f69-da63-421a-8733-0f13842bb3bc/download/BASE_ITS_OTU.csv.gz
    - BASE_18S_taxonomy.xlsx (XLSX)
      URL: https://data.bioplatforms.com/dataset/dbd99948-a97b-4c96-9ac1-88e3096545c1/resource/1526f7fc-1784-4f9f-8d0e-b18f383580fc/download/BASE_18S

In [12]:
# Search for Australian Microbiome sample metadata
search_url = urljoin(API_URL, 'package_search')
params = {
    'q': 'Australian Microbiome sample metadata',
    'rows': 10
}
resp = requests.get(search_url, params=params)
if resp.status_code == 200:
    data = resp.json()
    for ds in data.get('result', {}).get('results', []):
        title = ds.get('title', '?')
        name = ds.get('name', '?')
        print(f"\n{title}")
        for r in ds.get('resources', []):
            rname = r.get('name', '?')
            rformat = r.get('format', '?')
            if any(kw in rname.lower() for kw in ['metadata', 'contextual', 'sample', 'env']):
                rurl = r.get('url', '?')
                print(f"  → {rname} ({rformat})")
                print(f"    {rurl[:250]}")


Australian Microbiome Contextual Metadata
  → Contextual Metadata Matrix (XLSX)
    https://data.bioplatforms.com/dataset/f3e0eee9-600a-4222-8dc6-b1f997e00b93/resource/0a6d1159-dcb4-4e05-8cd7-17a185764c14/download/AM_Contextual_Data_Master_Sheet-20180501.xlsx

BASE_sample_collection_Dec18
  → base_sample_collection_dec18 (PDF)
    https://data.bioplatforms.com/dataset/2e85910e-8722-4b11-b130-4b13ad8577fc/resource/25d31915-0252-4109-89eb-3f2d62aceeaa/download/BASE_sample_collection_Dec18.pdf

Plant Pathogen, Genomics, Illumina Shortread, Sample ID 395150

Plant Pathogen, Genomics, Illumina Shortread, Sample ID 395171

Plant Pathogen, Genomics, Illumina Shortread, Sample ID 395225

Plant Pathogen, Genomics, Illumina Shortread, Sample ID 395224

Plant Pathogen, Genomics, Illumina Shortread, Sample ID 395213

Plant Pathogen, Genomics, Illumina Shortread, Sample ID 395212

Plant Pathogen, Genomics, Illumina Shortread, Sample ID 395211

Plant Pathogen, Genomics, Illumina Shortread, Sample I

In [14]:
# Check what's actually in the downloaded file
print("First 500 bytes of the downloaded file:")
with open(META_PATH, 'rb') as f:
    content = f.read(500)
    print(content)

# Try to detect the actual file type
print(f"\nFile size: {len(content)} bytes")
if content.startswith(b'PK'):
    print("Looks like a ZIP/XLSX file")
elif content.startswith(b'<!DOCTYPE') or content.startswith(b'<html'):
    print("Looks like HTML — download failed or redirected")
elif content.startswith(b'\xd0\xcf\x11\xe0'):
    print("Looks like old-format XLS file")
else:
    print("Unknown format")

# Also try the metadata as CSV — Australian Microbiome often provides CSVs too
# Let's search for alternative metadata formats
search_url = urljoin(API_URL, 'package_search')
params = {
    'q': 'Australian Microbiome contextual metadata sample',
    'rows': 5
}
resp = requests.get(search_url, params=params)
if resp.status_code == 200:
    data = resp.json()
    for ds in data.get('result', {}).get('results', []):
        for r in ds.get('resources', []):
            rname = r.get('name', '?')
            rformat = r.get('format', '?')
            if any(ext in rname.lower() for ext in ['.csv', '.tsv', '.txt', '.gz']):
                print(f"\nAlternative: {rname} ({rformat})")
                print(f"  {r.get('url', '?')[:250]}")

First 500 bytes of the downloaded file:
b'<!doctype html>\n<html lang="en">\n  <head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <title>\n      \n        Login | BioCommons Access\n      \n    </title>\n\n    <link\n      rel="icon"\n      href="https://images.squarespace-cdn.com/content/5d3a4213cf4f5b00014ea1db/0b7ee000-0f7e-4ca3-a28a-83c52b057f04/Australian-Biocommons-Favicon-RGB_cropped.png?format=100w&content-type=image%2Fpng"\n      type="image/x-icon"\n    >\n\n        <meta charset'

File size: 500 bytes
Unknown format



Alternative: 139590_MGE_H72YYDSX2_AACCATAGAA-CCATCTCGCC_S191_L003_R1_001.fastq.gz (FASTQ)
  https://data.bioplatforms.com/dataset/bpa-amdb-metagenomics-novaseq-139590-h72yydsx2/resource/24d9ee47dd941100a00d72574c80d0e6/download/139590_MGE_H72YYDSX2_AACCATAGAA-CCATCTCGCC_S191_L003_R1_001.fastq.gz

Alternative: 139590_MGE_H72YYDSX2_AACCATAGAA-CCATCTCGCC_S191_L003_R2_001.fastq.gz (FASTQ)
  https://data.bioplatforms.com/dataset/bpa-amdb-metagenomics-novaseq-139590-h72yydsx2/resource/23ef3dbc11e6c90759c96f6dedf69d05/download/139590_MGE_H72YYDSX2_AACCATAGAA-CCATCTCGCC_S191_L003_R2_001.fastq.gz

Alternative: 139590_MGE_H72YYDSX2_AACCATAGAA-CCATCTCGCC_S191_L004_R1_001.fastq.gz (FASTQ)
  https://data.bioplatforms.com/dataset/bpa-amdb-metagenomics-novaseq-139590-h72yydsx2/resource/ff101029f9c8fb2e6eeaa28d1e248c41/download/139590_MGE_H72YYDSX2_AACCATAGAA-CCATCTCGCC_S191_L004_R1_001.fastq.gz

Alternative: 139590_MGE_H72YYDSX2_AACCATAGAA-CCATCTCGCC_S191_L004_R2_001.fastq.gz (FASTQ)
  https://data.

In [15]:
# Clone the repo (small, mostly CSV/Excel files)
!git clone https://github.com/AusMicrobiome/contextualdb.git

Cloning into 'contextualdb'...


remote: Repository not found.
fatal: repository 'https://github.com/AusMicrobiome/contextualdb.git/' not found


In [16]:
# Download the OTU table in the background
OTU_URL = 'https://data.bioplatforms.com/dataset/dbd99948-a97b-4c96-9ac1-88e3096545c1/resource/3cb00190-c1dd-45ff-b8a2-f8f761fe1ce4/download/BASE_16S_OTU.csv.gz'
TAX_URL = 'https://data.bioplatforms.com/dataset/dbd99948-a97b-4c96-9ac1-88e3096545c1/resource/908fb8ab-4a0a-4b6c-a31d-79a9e422661b/download/BASE_16S_taxonomy.xlsx'

# These should download without authentication
for url, name in [(OTU_URL, 'BASE_16S_OTU.csv.gz'), (TAX_URL, 'BASE_16S_taxonomy.xlsx')]:
    output_path = DATA_DIR / name
    if not output_path.exists():
        resp = requests.get(url, stream=True)
        if resp.status_code == 200:
            with open(output_path, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=1048576):
                    f.write(chunk)
            print(f"Downloaded: {name} ({output_path.stat().st_size:,} bytes)")
        else:
            print(f"Failed: {name} (HTTP {resp.status_code})")

Downloaded: BASE_16S_OTU.csv.gz (6,466,316 bytes)


Downloaded: BASE_16S_taxonomy.xlsx (4,277,452 bytes)


In [19]:
# Search specifically for metadata/spreadsheet packages
search_terms = [
    'contextual metadata',
    'sample metadata spreadsheet', 
    'AM_contextual',
    'BASE sample metadata',
    'a16s metadata'
]

for term in search_terms:
    params = {'q': term, 'rows': 5}
    resp = requests.get(urljoin(API_URL, 'package_search'), params=params)
    if resp.status_code == 200:
        data = resp.json()
        count = data.get('result', {}).get('count', 0)
        if count > 0:
            print(f"\n'{term}' → {count} results")
            for ds in data['result']['results'][:3]:
                title = ds.get('title', '?')
                name = ds.get('name', '?')
                # Check for downloadable spreadsheet resources
                sheet_resources = [r for r in ds.get('resources', []) 
                                   if any(ext in r.get('name','').lower() or ext in r.get('format','').lower() 
                                         for ext in ['.xlsx', '.csv', '.tsv'])]
                if sheet_resources:
                    print(f"  {title}")
                    for r in sheet_resources[:3]:
                        print(f"    → {r['name']} ({r.get('format','?')})")


'contextual metadata' → 2191 results



'sample metadata spreadsheet' → 18541 results
  waddy, Population Genetics, Illumina FASTQ, Phyllode sample
    → TSI_AGRF_CAGRF230313975_22JM2GLT3_metadata.xlsx (XLSX)
    → TSI_22JM2GLT3_358847_samplemetadata_ingest.xlsx (XLSX)



'AM_contextual' → 49279 results



'BASE sample metadata' → 20565 results



'a16s metadata' → 35834 results


In [20]:
# Download the OTU table and taxonomy — these are public, no auth needed
OTU_URL = 'https://data.bioplatforms.com/dataset/dbd99948-a97b-4c96-9ac1-88e3096545c1/resource/3cb00190-c1dd-45ff-b8a2-f8f761fe1ce4/download/BASE_16S_OTU.csv.gz'
TAX_URL = 'https://data.bioplatforms.com/dataset/dbd99948-a97b-4c96-9ac1-88e3096545c1/resource/908fb8ab-4a0a-4b6c-a31d-79a9e422661b/download/BASE_16S_taxonomy.xlsx'

for url, name in [(OTU_URL, 'BASE_16S_OTU.csv.gz'), (TAX_URL, 'BASE_16S_taxonomy.xlsx')]:
    output_path = DATA_DIR / name
    if not output_path.exists():
        resp = requests.get(url, stream=True)
        if resp.status_code == 200:
            with open(output_path, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=1048576):
                    f.write(chunk)
            print(f"✅ {name}: {output_path.stat().st_size:,} bytes")
        else:
            print(f"❌ {name}: HTTP {resp.status_code}")
    else:
        print(f"✓ {name} already cached ({output_path.stat().st_size:,} bytes)")

✓ BASE_16S_OTU.csv.gz already cached (6,466,316 bytes)
✓ BASE_16S_taxonomy.xlsx already cached (4,277,452 bytes)


In [21]:
# Option 1: Try the BASE metadata from their public S3 bucket
# The Australian Microbiome data is also hosted on AWS S3 (no auth)
base_urls = [
    'https://base-public-data.s3.amazonaws.com/amplicons/metadata/',
    'https://public-data.bioplatforms.com/bpa/base/amplicons/',
    'https://downloads.bioplatforms.com/bpa/base/amplicons/'
]

for url in base_urls:
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            print(f"✅ {url} — accessible")
            # Print first 500 chars to see directory listing
            print(resp.text[:500])
        else:
            print(f"❌ {url} — HTTP {resp.status_code}")
    except Exception as e:
        print(f"❌ {url} — {e}")

❌ https://base-public-data.s3.amazonaws.com/amplicons/metadata/ — HTTP 404
❌ https://public-data.bioplatforms.com/bpa/base/amplicons/ — HTTPSConnectionPool(host='public-data.bioplatforms.com', port=443): Max retries exceeded with url: /bpa/base/amplicons/ (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7d81ee721090>: Failed to resolve 'public-data.bioplatforms.com' ([Errno -2] Name or service not known)"))


❌ https://downloads.bioplatforms.com/bpa/base/amplicons/ — HTTPSConnectionPool(host='downloads-qcif.bioplatforms.com', port=443): Max retries exceeded with url: /bpa/base/amplicons/ (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7d81ee7216d0>, 'Connection to downloads-qcif.bioplatforms.com timed out. (connect timeout=10)'))


In [22]:
# Option 2: Query ENA for Australian Microbiome samples with environment metadata
# This uses the ENA Portal API which is always open
sample_query = 'https://www.ebi.ac.uk/ena/portal/api/search?result=sample&query=tax_eq=2%20AND%20collection_date%3E%3D2015-01-01%20AND%20country=%22Australia%22&fields=sample_accession,sample_alias,sample_title,environment_biome,environment_feature,environment_material&format=tsv&limit=10'

resp = requests.get(sample_query)
if resp.status_code == 200:
    print("✅ ENA query successful")
    print(resp.text[:800])
else:
    print(f"ENA query failed: HTTP {resp.status_code}")

ENA query failed: HTTP 400


In [23]:
# Query ENA for all Australian Microbiome samples with environment metadata
# The Australian Microbiome BioProjects we confirmed are NOT in MicrobeAtlas
# We'll query by project accession

import time

aus_bioprojects = [
    'PRJNA317932', 'PRJNA509678', 'PRJEB25028', 'PRJEB25787',
    'PRJEB26196', 'PRJNA509680', 'PRJEB20995'
]

all_samples = []

for bp in aus_bioprojects:
    # ENA Portal API — returns all samples for a BioProject with environment fields
    query = (
        f'https://www.ebi.ac.uk/ena/portal/api/search'
        f'?result=sample'
        f'&query=secondary_study_accession%3D{bp}'
        f'&fields=sample_accession,sample_alias,sample_title,'
        f'environment_biome,environment_feature,environment_material,'
        f'collection_date,country,location'
        f'&format=tsv&limit=50000'
    )
    
    print(f"Querying {bp}...")
    try:
        resp = requests.get(query, timeout=30)
        if resp.status_code == 200 and len(resp.text) > 100:
            # Parse TSV
            lines = resp.text.strip().split('\n')
            if len(lines) > 1:  # header + data
                header = lines[0].split('\t')
                for line in lines[1:]:
                    values = line.split('\t')
                    sample_data = dict(zip(header, values))
                    sample_data['bioproject'] = bp
                    all_samples.append(sample_data)
                print(f"  ✅ {len(lines)-1} samples")
            else:
                print(f"  ⚠️ No samples found")
        else:
            print(f"  ❌ HTTP {resp.status_code}")
    except Exception as e:
        print(f"  ❌ Error: {e}")
    time.sleep(0.5)

# Save results
meta_ena = pd.DataFrame(all_samples)
meta_ena.to_csv(DATA_DIR / 'aus_ena_metadata.csv', index=False)

print(f'\nTotal samples: {len(meta_ena)}')
print(f'Unique bioprojects: {meta_ena["bioproject"].nunique()}')
print(f'\nEnvironment biomes:')
print(meta_ena['environment_biome'].value_counts().head(15).to_string())

Querying PRJNA317932...


  ❌ HTTP 500


Querying PRJNA509678...


  ❌ HTTP 500


Querying PRJEB25028...


...

  ❌ HTTP 500


Querying PRJEB20995...


  ❌ HTTP 500



Total samples: 0


KeyError: 'bioproject'

In [24]:
# Check if sample IDs contain environment information
otu = pd.read_csv(DATA_DIR / 'BASE_16S_OTU.csv.gz', nrows=5)
sample_cols = [c for c in otu.columns if c not in ['#OTU ID', 'OTU ID', 'taxonomy']]
print(f"Sample columns: {len(sample_cols)}")
print("First 10 sample IDs:")
for s in sample_cols[:10]:
    print(f"  {s}")

Sample columns: 1024
First 10 sample IDs:
  OTU_Id
  12520
  12521
  12522
  12523
  12524
  12525
  12526
  12527
  12528


In [25]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data/aus_microbiome')

# Verify the metadata file
meta_path = DATA_DIR / 'AM_Contextual_Data_Master_Sheet-20180501.xlsx'
print(f"Metadata file: {meta_path.exists()}")
print(f"Size: {meta_path.stat().st_size:,} bytes")

# Inspect the structure
meta = pd.read_excel(meta_path, engine='openpyxl')
print(f"\nShape: {meta.shape[0]:,} samples × {meta.shape[1]} columns")
print(f"Columns: {meta.columns.tolist()[:30]}")

# Find environment classification columns
env_candidates = [c for c in meta.columns if any(kw in c.lower() for kw in 
    ['env', 'sample_type', 'biome', 'habitat', 'broad', 'ecosystem', 'depth', 'soil', 'water', 'marine', 'description', 'amplicon'])]
print(f"\nEnvironment-related columns: {env_candidates}")

# Also check for sample ID columns that would match the OTU table
id_candidates = [c for c in meta.columns if any(kw in c.lower() for kw in 
    ['sample', 'barcode', 'id', 'accession', 'run', 'sequencing', 'amplicon', 'base'])]
print(f"\nSample ID columns: {id_candidates}")

# Print first few rows of key columns
sample_cols = id_candidates[:5] + env_candidates[:5] if id_candidates and env_candidates else meta.columns[:10].tolist()
print(f"\nFirst 3 rows of sample + env columns:")
print(meta[sample_cols].head(3).to_string())

Metadata file: True
Size: 2,078,913 bytes



Shape: 1,669 samples × 64 columns
Columns: ['Sample_ID', 'Date sampled', 'latitude', 'longitude', 'Depth', 'Horizon', 'soil sample storage method', 'geo_loc', 'location description', 'broad land use', 'Detailed land use', 'General Ecological Zone', 'Vegetation Type', 'Vegetation Total cover (%)', 'Vegetation Dom. Trees (%)', 'Vegetation Dom. Shrubs (%)', 'Vegetation Dom. Grasses (%)', 'Elevation ()', 'Slope (%)', 'Slope Aspect (Direction or degrees; e.g., NW or 315°)', 'Profile Position controlled vocab (5)', 'Australian Soil Classification controlled vocab (6)', 'FAO soil classification controlled vocab (7)', 'Immediate Previous Land Use controlled vocab (2)', 'Date since change in Land Use', 'Crop rotation 1yr since present', 'Crop rotation 2yrs since present', 'Crop rotation 3yrs since present', 'Crop rotation 4yrs since present', 'Crop rotation 5yrs since present']

Environment-related columns: ['Depth', 'soil sample storage method', 'location description', 'broad land use', 'Aust

In [26]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data/aus_microbiome')

# Check OTU table structure
otu_path = DATA_DIR / 'BASE_16S_OTU.csv.gz'
print("Loading OTU table (first 5 rows)...")
otu = pd.read_csv(otu_path, nrows=5)

print(f"\nOTU table columns: {len(otu.columns)}")
print(f"First 5 column names: {otu.columns[:5].tolist()}")
print(f"Last 5 column names: {otu.columns[-5:].tolist()}")

# Check if the first column is OTU IDs
first_col = otu.columns[0]
print(f"\nFirst column name: '{first_col}'")
print(f"First 5 OTU IDs: {otu[first_col].head().tolist()}")

# Check if there's a taxonomy column
tax_cols = [c for c in otu.columns if any(kw in c.lower() for kw in ['taxonomy', 'tax', 'otu', 'sequence'])]
print(f"\nTaxonomy-related columns: {tax_cols}")

# Check sample ID range
sample_cols = [c for c in otu.columns if c not in [first_col] + tax_cols]
print(f"\nSample columns: {len(sample_cols)}")
print(f"Sample ID range: {sample_cols[0]} to {sample_cols[-1]}")
print(f"First 10 sample IDs: {sample_cols[:10]}")

# Check if any sample IDs look like the metadata Sample_ID format
meta_ids = set(pd.read_excel(DATA_DIR / 'AM_Contextual_Data_Master_Sheet-20180501.xlsx', 
                             usecols=['Sample_ID'], engine='openpyxl')['Sample_ID'].dropna().astype(str))
print(f"\nMetadata Sample_ID examples: {list(meta_ids)[:5]}")
print(f"Total metadata samples: {len(meta_ids)}")

# Check for overlap between OTU column names and metadata IDs
otu_sample_ids = set(str(c) for c in sample_cols)
overlap = otu_sample_ids & meta_ids
print(f"Direct overlap between OTU columns and metadata Sample_IDs: {len(overlap)}")
if overlap:
    print(f"Overlap examples: {list(overlap)[:5]}")
else:
    # Check if OTU columns are numeric and metadata IDs contain numbers
    otu_numeric = [c for c in sample_cols if str(c).isdigit()]
    print(f"OTU has {len(otu_numeric)} numeric sample columns")
    meta_numeric = [m for m in meta_ids if any(str(n) in m for n in otu_numeric[:5])]
    print(f"Metadata IDs containing OTU sample numbers: {len(meta_numeric)}")

Loading OTU table (first 5 rows)...

OTU table columns: 1024
First 5 column names: ['OTU_Id', '12520', '12521', '12522', '12523']
Last 5 column names: ['12515', '12516', '12517', '12518', '12519']

First column name: 'OTU_Id'
First 5 OTU IDs: ['16S_OTUa_4', '16S_OTUa_1', '16S_OTUa_3', '16S_OTUa_2', '16S_OTUa_5']

Taxonomy-related columns: ['OTU_Id']

Sample columns: 1023
Sample ID range: 12520 to 12519
First 10 sample IDs: ['12520', '12521', '12522', '12523', '12524', '12525', '12526', '12527', '12528', '12529']



Metadata Sample_ID examples: ['102.100.100/15959', '102.100.100/13449', '102.100.100/19270', '102.100.100/19260', '102.100.100/7086']
Total metadata samples: 1665
Direct overlap between OTU columns and metadata Sample_IDs: 0
OTU has 1023 numeric sample columns
Metadata IDs containing OTU sample numbers: 5


In [27]:
# Check what other files are available in the BASE OTUs dataset
import requests

# The BASE OTUs dataset we got the OTU table from
base_otu_dataset = 'https://data.bioplatforms.com/api/3/action/package_show?id=base-otus'
resp = requests.get(base_otu_dataset)
if resp.status_code == 200:
    resources = resp.json()['result']['resources']
    print("All resources in BASE OTUs dataset:")
    for r in resources:
        name = r.get('name', '?')
        fmt = r.get('format', '?')
        url = r.get('url', '?')
        print(f"  {name} ({fmt})")
        # Flag potential mapping files
        if any(kw in name.lower() for kw in ['map', 'sample', 'link', 'key', 'lookup', 'barcode']):
            print(f"    ⚠ POTENTIAL MAPPING FILE: {url[:200]}")

All resources in BASE OTUs dataset:
  BASE_16S_OTU.csv.gz (CSV)
  BASE_18S_OTU.csv.gz (CSV)
  BASE_A16S_OTU.csv.gz (CSV)
  BASE_ITS_OTU.csv.gz (CSV)
  BASE_18S_taxonomy.xlsx (XLSX)
  BASE_ITS_taxonomy.xlsx (XLSX)
  BASE_A16S_taxonomy.xlsx (XLSX)
  BASE_16S_taxonomy.xlsx (XLSX)
  BASE_A16S_OTU.fasta ()
  BASE_ITS_OTU.fasta ()
  BASE_18S_OTU.fasta ()
  BASE_16S_OTU.fasta ()


In [28]:
import requests
import xml.etree.ElementTree as ET

# Check if numeric OTU IDs map to ENA run accessions (ERR/SRR + number)
# Test a few sample IDs against ENA
test_ids = ['12520', '12521', '13540', '13541', '13542']
print("Testing if OTU numeric IDs correspond to ENA run accessions...")
for sid in test_ids[:3]:
    for prefix in ['ERR', 'SRR']:
        url = f'https://www.ebi.ac.uk/ena/browser/api/xml/{prefix}{sid}'
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                root = ET.fromstring(resp.text)
                # Extract sample alias if available
                for sample in root.iter('SAMPLE'):
                    alias = sample.findtext('SAMPLE_NAME/TAXON_ID', '') or sample.findtext('alias', '')
                print(f"  {prefix}{sid}: FOUND in ENA")
        except:
            pass

# Search Bioplatforms CKAN for sample mapping files
print("\nSearching Bioplatforms for sample mapping files...")
search_terms = ['BASE sample map', 'amplicon sample number', 'BPA sample index', 'sample lookup']
for term in search_terms:
    url = 'https://data.bioplatforms.com/api/3/action/package_search'
    params = {'q': term, 'rows': 3}
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            count = data.get('result', {}).get('count', 0)
            if count > 0:
                print(f"\n'{term}' → {count} results")
                for ds in data['result']['results'][:2]:
                    print(f"  {ds.get('title','?')[:120]}")
    except:
        pass

# Check if there's an AMPLICON_METADATA or similar file in the Bioplatforms API
print("\nSearching for amplicon metadata...")
for term in ['AMPLICON_METADATA', 'A16S sample number', 'BASE sample number mapping']:
    url = 'https://data.bioplatforms.com/api/3/action/package_search'
    params = {'q': term, 'rows': 3}
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            count = data.get('result', {}).get('count', 0)
            if count > 0:
                print(f"\n'{term}' → {count} results")
                for ds in data['result']['results'][:3]:
                    print(f"  {ds.get('title','?')[:120]}")
                    for r in ds.get('resources', []):
                        if any(ext in r.get('name','').lower() or ext in r.get('format','').lower() 
                               for ext in ['.csv', '.tsv', '.txt', '.xlsx']):
                            print(f"    → {r.get('name','?')[:100]}")
    except:
        pass

Testing if OTU numeric IDs correspond to ENA run accessions...



Searching Bioplatforms for sample mapping files...



'BASE sample map' → 836 results
  Quokka, Sample Images
  Quokka, Sample Images



'amplicon sample number' → 8682 results
  Plant Pathogen, Genomics, Illumina Shortread, Sample ID 394855
  Plant Pathogen, Genomics, PacBio HiFi, Sample ID 394855



'BPA sample index' → 23438 results
  20170420_Analysed data_Metabolomics_BPA_SC_MA_20170220
  20170419_Analysed data_Metabolomics_BPA_SC_MA_20170112



Searching for amplicon metadata...



'AMPLICON_METADATA' → 65144 results
  Metadata: PPA_Faba bean_field_trial_2022_soil_&_environmental_metadata
  Australian Microbiome Contextual Metadata
  Antibiotic Resistant Sepsis Pathogens sample metadata



'A16S sample number' → 19393 results
  Marine Microbes Amplicons A16S 102.100.100/34369 AVFAA
  Plant Pathogen, Genomics, Illumina Shortread, Sample ID 394855
  Plant Pathogen, Genomics, PacBio HiFi, Sample ID 394855
    → 394855_PP_AGRF_DA218825_ccs_statistics.csv



'BASE sample number mapping' → 24 results
  Double-barred finch, Reference genome, PacBio-HiFi, blood
  Double-barred finch, Reference genome, PacBio-HiFi, blood
  Prostrate flame pea, Population Genetics, Illumina FASTQ, Leaf
    → TSI_AGRF_HTF5VDRX2_metadata_358809.xlsx
    → TSI_HTF5VDRX2_358809_samplemetadata_ingest.xlsx


In [29]:
import requests
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data/aus_microbiome')

# Load the OTU table to get the numeric sample IDs
otu = pd.read_csv(DATA_DIR / 'BASE_16S_OTU.csv.gz', nrows=2)
# Get all sample columns (everything except OTU_Id)
sample_ids = [c for c in otu.columns if c != 'OTU_Id']
print(f"OTU sample IDs: {len(sample_ids)} columns")
print(f"Range: {sample_ids[0]} to {sample_ids[-1]}")

# Check if these numeric IDs correspond to CKAN package identifiers
# by querying a few directly
test_ids = sample_ids[:3] + sample_ids[-3:]
print(f"\nTesting if numeric IDs are CKAN package identifiers...")
for sid in test_ids:
    url = f'https://data.bioplatforms.com/api/3/action/package_show?id={sid}'
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            title = resp.json()['result'].get('title', '?')
            print(f"  {sid}: FOUND → {title[:150]}")
        else:
            print(f"  {sid}: HTTP {resp.status_code}")
    except Exception as e:
        print(f"  {sid}: Error - {e}")

# Also try the BPA ID format with the numeric IDs
print(f"\nTesting if numeric IDs match BPA sample IDs...")
for sid in test_ids:
    # Try the BPA ID format seen in metadata: 102.100.100/XXXXX
    # But maybe the numeric part is our sample IDs?
    bpa_id = f"102.100.100/{sid}"
    url = f'https://data.bioplatforms.com/api/3/action/package_show?id={bpa_id}'
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            title = resp.json()['result'].get('title', '?')
            print(f"  {bpa_id}: FOUND → {title[:150]}")
    except:
        pass

OTU sample IDs: 1023 columns
Range: 12520 to 12519

Testing if numeric IDs are CKAN package identifiers...


  12520: HTTP 404


  12521: HTTP 404


  12522: HTTP 404


  12517: HTTP 404


  12518: HTTP 404


  12519: HTTP 404

Testing if numeric IDs match BPA sample IDs...


In [30]:
import requests
import time

# ENA Portal API — search for 16S amplicon studies from 2025-2026
# filtered by size (large studies with many runs)
queries = [
    # Large soil/metagenomic 16S studies from 2025
    ('https://www.ebi.ac.uk/ena/portal/api/search'
     '?result=study'
     '&query=first_public%3E%3D2025-01-01%20AND%20tax_eq%3D2%20AND%20library_source%3DMETAGENOMIC%20AND%20library_strategy%3DAMPLICON'
     '&fields=study_accession,secondary_study_accession,study_title,study_description,first_public,last_updated'
     '&format=tsv&limit=200'),
    
    # Large 16S studies from 2025 by run count
    ('https://www.ebi.ac.uk/ena/portal/api/search'
     '?result=study'
     '&query=first_public%3E%3D2025-01-01%20AND%20tax_eq%3D2%20AND%20library_strategy%3DAMPLICON'
     '&fields=study_accession,secondary_study_accession,study_title,first_public'
     '&format=tsv&limit=500'),
]

for url in queries:
    print(f"\nQuerying ENA...")
    try:
        resp = requests.get(url, timeout=30)
        if resp.status_code == 200:
            lines = resp.text.strip().split('\n')
            print(f"Found {len(lines)-1} studies")
            # Print first 20
            for line in lines[:21]:
                print(line[:200])
    except Exception as e:
        print(f"Error: {e}")
    time.sleep(1)

# Also search specifically for soil microbiome + 16S + 2025
specific_url = (
    'https://www.ebi.ac.uk/ena/portal/api/search'
    '?result=study'
    '&query=first_public%3E%3D2025-01-01%20AND%20tax_eq%3D2%20AND%20library_strategy%3DAMPLICON%20AND%20(soil%20OR%20marine%20OR%20freshwater%20OR%20rhizosphere)'
    '&fields=study_accession,secondary_study_accession,study_title,first_public'
    '&format=tsv&limit=200'
)
resp = requests.get(specific_url, timeout=30)
if resp.status_code == 200:
    lines = resp.text.strip().split('\n')
    print(f"\nEnvironmental 16S studies 2025+: {len(lines)-1}")
    for line in lines[:30]:
        print(line[:200])


Querying ENA...



Querying ENA...


In [31]:
import requests

# Try the ENA Browser API (XML) for a specific study type
# Search for recent soil 16S studies via NCBI SRA instead
print("=== Searching NCBI SRA for recent soil 16S studies ===")

# NCBI SRA search via Entrez
search_url = (
    'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
    '?db=sra'
    '&term=(soil+OR+marine+OR+freshwater)+AND+16S+AND+amplicon+AND+("2025"[Publication+Date]+:"2026"[Publication+Date])'
    '&retmax=50'
    '&usehistory=y'
    '&retmode=json'
)

try:
    resp = requests.get(search_url, timeout=30)
    if resp.status_code == 200:
        data = resp.json()
        count = data['esearchresult']['count']
        ids = data['esearchresult']['idlist']
        print(f"Found {count} SRA entries")
        print(f"First 20 IDs: {ids[:20]}")
        
        # Get summaries for the first 10
        if ids:
            summary_url = (
                'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi'
                f'?db=sra&id={",".join(ids[:10])}&retmode=json'
            )
            summary_resp = requests.get(summary_url, timeout=30)
            if summary_resp.status_code == 200:
                summary_data = summary_resp.json()
                for uid in ids[:10]:
                    rec = summary_data['result'].get(uid, {})
                    title = rec.get('title', '?')
                    bioproject = rec.get('bioproject', '?')
                    runs = rec.get('runs', '?')
                    print(f"\n  {uid}")
                    print(f"  Title: {title[:150]}")
                    print(f"  BioProject: {bioproject}")
                    print(f"  Runs: {runs}")
    else:
        print(f"HTTP {resp.status_code}: {resp.text[:300]}")
except Exception as e:
    print(f"Error: {e}")

=== Searching NCBI SRA for recent soil 16S studies ===


Found 288982 SRA entries
First 20 IDs: ['45413434', '45413432', '45413413', '45413397', '45413382', '45413325', '45413187', '45413176', '45413137', '45413136', '45413098', '45413075', '45413057', '45413056', '45413040', '45413027', '45412924', '45412923', '45412922', '45412914']

  45413434
  Title: ?
  BioProject: ?
  Runs:                                 <Run acc="ERR17328073" total_spots="" total_bases="" load_done="false" is_public="true" cluster_name="" unavailable="true"/>                                

  45413432
  Title: ?
  BioProject: ?
  Runs:                                 <Run acc="ERR17328134" total_spots="" total_bases="" load_done="false" is_public="true" cluster_name="" unavailable="true"/>                                

  45413413
  Title: ?
  BioProject: ?
  Runs:                                 <Run acc="ERR17327925" total_spots="" total_bases="" load_done="false" is_public="true" cluster_name="" unavailable="true"/>                                

  45413397


In [33]:
import requests
import json

# Search NCBI SRA for recent large 16S studies
search_url = (
    'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
    '?db=sra'
    '&term=((soil+OR+marine+OR+freshwater)+AND+16S+AND+amplicon+AND+("2025"[Publication+Date]+:"2026"[Publication+Date]))'
    '&retmax=30&usehistory=y&retmode=json'
)

resp = requests.get(search_url, timeout=30)
if resp.status_code == 200:
    data = resp.json()
    ids = data['esearchresult']['idlist']
    print(f"Found {data['esearchresult']['count']} studies, retrieved {len(ids)} IDs")
    
    if ids:
        # Get summaries
        summary_url = (
            'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi'
            f'?db=sra&id={",".join(ids[:20])}&retmode=json'
        )
        sresp = requests.get(summary_url, timeout=30)
        if sresp.status_code == 200:
            sdata = sresp.json()
            candidates = []
            for uid in ids[:20]:
                rec = sdata['result'].get(uid, {})
                title = rec.get('title', '?')
                bp = rec.get('bioproject', '?')
                runs = rec.get('runs', '?')
                # Parse run count
                run_count = 0
                try:
                    run_count = int(runs) if runs and runs != '?' else 0
                except:
                    pass
                
                if run_count > 50:  # Focus on larger studies
                    candidates.append((uid, bp, title[:200], run_count))
                    print(f"\n{uid} | {bp} | runs={runs}")
                    print(f"  {title[:200]}")
            
            # Save candidates
            with open('../data/candidate_datasets_2025.json', 'w') as f:
                json.dump(candidates, f, indent=2)
            print(f"\nSaved {len(candidates)} candidate studies with >50 runs")
else:
    print(f"Search failed: HTTP {resp.status_code}")

Found 288982 studies, retrieved 30 IDs

Saved 0 candidate studies with >50 runs


In [34]:
import requests
import xml.etree.ElementTree as ET

# Better approach: Search BioProjects directly via NCBI
# Look for 16S amplicon BioProjects from 2025-2026 with many samples
search_url = (
    'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
    '?db=bioproject'
    '&term=((soil+OR+marine+OR+freshwater)+AND+16S+AND+amplicon)'
    '&retmax=30&usehistory=y&retmode=json'
)

print("Searching BioProjects...")
resp = requests.get(search_url, timeout=30)
if resp.status_code == 200:
    data = resp.json()
    bp_ids = data['esearchresult']['idlist']
    print(f"Found {data['esearchresult']['count']} BioProjects, retrieved {len(bp_ids)} IDs\n")
    
    candidates = []
    for bp_id in bp_ids[:20]:
        # Get BioProject summary
        summary_url = (
            'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi'
            f'?db=bioproject&id={bp_id}&retmode=json'
        )
        sresp = requests.get(summary_url, timeout=30)
        if sresp.status_code == 200:
            sdata = sresp.json()
            rec = sdata['result'].get(bp_id, {})
            title = rec.get('project_title', rec.get('title', '?'))
            accession = rec.get('project_accession', '?')
            
            # Get sample count from BioSample link
            # Query linked BioSamples
            link_url = (
                'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi'
                f'?dbfrom=bioproject&db=biosample&id={bp_id}&retmode=json'
            )
            lresp = requests.get(link_url, timeout=30)
            sample_count = 0
            if lresp.status_code == 200:
                ldata = lresp.json()
                links = ldata.get('linksets', [{}])[0].get('linksetdbs', [{}])[0].get('links', [])
                sample_count = len(links) if links else 0
            
            if sample_count >= 50:  # Focus on larger studies
                candidates.append((bp_id, accession, title[:200], sample_count))
                print(f"{accession} | {sample_count} samples")
                print(f"  {title[:200]}\n")
    
    print(f"\nFound {len(candidates)} candidate BioProjects with ≥50 samples")
else:
    print(f"Search failed: HTTP {resp.status_code}")

# Also try ENA for studies with many runs
print("\n=== ENA large 16S studies 2025+ ===")
ena_url = (
    'https://www.ebi.ac.uk/ena/portal/api/search'
    '?result=study'
    '&query=first_public%3E%3D2025-01-01%20AND%20library_strategy%3DAMPLICON%20AND%20library_source%3DMETAGENOMIC'
    '&fields=study_accession,secondary_study_accession,study_title,run_count,first_public'
    '&format=tsv&limit=50'
)
try:
    ena_resp = requests.get(ena_url, timeout=30)
    if ena_resp.status_code == 200:
        lines = ena_resp.text.strip().split('\n')
        print(f"Found {len(lines)-1} studies")
        # Print those with many runs
        for line in lines[:30]:
            parts = line.split('\t')
            if len(parts) >= 4:
                title = parts[2] if len(parts) > 2 else ''
                runs = parts[3] if len(parts) > 3 else '0'
                try:
                    if int(runs) >= 50:
                        print(f"  {parts[0]} | runs={runs} | {title[:150]}")
                except:
                    pass
except Exception as e:
    print(f"ENA error: {e}")

Searching BioProjects...
Found 1822 BioProjects, retrieved 30 IDs



? | 180 samples
  16S rRNA gene amplicon sequencing of PAH-degrading bacterial communities across heterogeneous coking soils



? | 487 samples
  The common symbiosis pathway controls plant root microbiomes in a host-specific manner



? | 160 samples
  Bacterial and fungal microbiome profiling of peach gummosis using 16S rRNA and ITS2 amplicon sequencing



? | 816 samples
  Metagenomic sequencing of microbial communities in the water conditioning tank of an artificial seawater aquarium (SEALIFE Nagoya)



? | 88 samples
  Effect of fertilization and plant growth stages on rhizosphere bacterial diversity of Sorghum bicolor cultivated in Nigeria.




Found 5 candidate BioProjects with ≥50 samples

=== ENA large 16S studies 2025+ ===


In [37]:
import requests
import xml.etree.ElementTree as ET
spark = get_spark_session()
# The five candidate BioProjects from the search results
# Let's get their proper accessions and check MicrobeAtlas overlap
candidates = [
    ('PRJNA1187703', 'PAH-degrading bacteria in coking soils', 180),
    ('PRJNA1161244', 'Common symbiosis pathway controls root microbiomes', 487),
    ('PRJNA1140468', 'Peach gummosis profiling', 160),
    ('PRJNA1095698', 'Aquarium water microbial communities', 816),
    ('PRJNA1105497', 'Sorghum rhizosphere diversity', 88),
]

print("=== Checking MicrobeAtlas overlap ===")
for bp, title, samples in candidates:
    # Check if this BioProject is in MicrobeAtlas
    result = spark.sql(f"""
        SELECT COUNT(*) AS n
        FROM arkinlab_microbeatlas.sample_metadata
        WHERE SampleID LIKE '%{bp}%'
           OR Project LIKE '%{bp}%'
    """).collect()
    n = result[0][0]
    status = "❌ IN MicrobeAtlas" if n > 0 else "✅ CLEAN"
    print(f"{bp}: {n} samples in MicrobeAtlas — {status}")
    if n == 0:
        print(f"  {title} ({samples} samples)")

# Also try to verify these are real BioProjects with data
print("\n=== Verifying BioProject accessions ===")
for bp, title, samples in candidates[:3]:  # Check top 3
    try:
        url = f'https://www.ebi.ac.uk/ena/browser/api/xml/{bp}'
        resp = requests.get(url, timeout=15)
        if resp.status_code == 200:
            print(f"{bp}: ✅ ENA record exists")
        else:
            print(f"{bp}: HTTP {resp.status_code} — may not be correct accession")
    except Exception as e:
        print(f"{bp}: Error - {e}")

=== Checking MicrobeAtlas overlap ===


PRJNA1187703: 0 samples in MicrobeAtlas — ✅ CLEAN
  PAH-degrading bacteria in coking soils (180 samples)


PRJNA1161244: 0 samples in MicrobeAtlas — ✅ CLEAN
  Common symbiosis pathway controls root microbiomes (487 samples)


PRJNA1140468: 0 samples in MicrobeAtlas — ✅ CLEAN
  Peach gummosis profiling (160 samples)


PRJNA1095698: 0 samples in MicrobeAtlas — ✅ CLEAN
  Aquarium water microbial communities (816 samples)


PRJNA1105497: 0 samples in MicrobeAtlas — ✅ CLEAN
  Sorghum rhizosphere diversity (88 samples)

=== Verifying BioProject accessions ===


PRJNA1187703: HTTP 404 — may not be correct accession


PRJNA1161244: HTTP 404 — may not be correct accession


PRJNA1140468: HTTP 404 — may not be correct accession


In [39]:
import requests
import xml.etree.ElementTree as ET

# Search NCBI BioProject directly for 16S amplicon studies from 2025
# with environmental keywords, filtering for those with linked BioSamples
search_url = (
    'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
    '?db=bioproject'
    '&term=((soil+OR+marine+OR+freshwater+OR+rhizosphere+OR+groundwater)+AND+16S+AND+amplicon+AND+("2025/01/01"[Publication+Date]+:"2026/12/31"[Publication+Date]))'
    '&retmax=30&usehistory=y&retmode=json'
)

print("Searching BioProject database (not SRA)...")
resp = requests.get(search_url, timeout=30)
if resp.status_code == 200:
    data = resp.json()
    bp_ids = data['esearchresult']['idlist']
    print(f"Found {data['esearchresult']['count']} BioProjects, retrieved {len(bp_ids)}\n")
    
    candidates = []
    for bp_id in bp_ids[:15]:
        # Fetch BioProject summary
        fetch_url = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=bioproject&id={bp_id}&retmode=json'
        fresp = requests.get(fetch_url, timeout=30)
        if fresp.status_code == 200:
            fdata = fresp.json()
            rec = fdata['result'].get(bp_id, {})
            title = rec.get('project_title', rec.get('title', '?'))
            accession = rec.get('project_accession', rec.get('accession', '?'))
            
            # Get linked BioSample count
            link_url = (
                'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi'
                f'?dbfrom=bioproject&db=biosample&id={bp_id}&retmode=json'
            )
            lresp = requests.get(link_url, timeout=30)
            sample_count = 0
            if lresp.status_code == 200:
                ldata = lresp.json()
                links = ldata.get('linksets', [{}])[0].get('linksetdbs', [{}])[0].get('links', [])
                sample_count = len(links) if links else 0
            
            if sample_count >= 50:
                candidates.append((accession, bp_id, title, sample_count))
                print(f"{accession} | {sample_count} samples")
                print(f"  {title[:200]}\n")
    
    print(f"\nFound {len(candidates)} candidate BioProjects with ≥50 samples")
    
    if candidates:
        # Check top candidates against MicrobeAtlas
        print("\n=== Checking MicrobeAtlas overlap ===")
        for accession, bp_id, title, samples in candidates[:5]:
            result = spark.sql(f"""
                SELECT COUNT(*) AS n
                FROM arkinlab_microbeatlas.sample_metadata
                WHERE Project LIKE '%{accession}%'
                   OR SampleID LIKE '%{accession}%'
            """).collect()
            n = result[0][0]
            status = "✅ CLEAN" if n == 0 else f"❌ {n} samples IN MicrobeAtlas"
            print(f"{accession}: {status} | {samples} samples | {title[:120]}")

Searching BioProject database (not SRA)...


Found 0 BioProjects, retrieved 0


Found 0 candidate BioProjects with ≥50 samples


In [40]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data/aus_microbiome')

# 1. Load OTU table
print("Loading OTU table...")
otu = pd.read_csv(DATA_DIR / 'BASE_16S_OTU.csv.gz', compression='gzip')
print(f"  {otu.shape[0]:,} OTUs × {otu.shape[1]:,} columns")

# 2. Load taxonomy and map OTUs to genera
print("Loading taxonomy...")
tax = pd.read_excel(DATA_DIR / 'BASE_16S_taxonomy.xlsx', engine='openpyxl')
# The taxonomy file has OTU IDs in first column and taxonomy strings
# Parse SILVA taxonomy to extract genus
def extract_genus(tax_string):
    if pd.isna(tax_string):
        return 'Unknown'
    for part in str(tax_string).split(';'):
        part = part.strip()
        if part.startswith('g__') and len(part) > 3:
            genus = part[3:]
            if genus and genus != 'uncultured' and 'Unknown' not in genus:
                return genus
    return 'Unknown'

# Build OTU → genus mapping
# The taxonomy file columns: first is OTU ID, second is taxonomy string
otu_to_genus = {}
for _, row in tax.iterrows():
    otu_id = str(row.iloc[0])
    tax_str = str(row.iloc[1]) if len(row) > 1 else str(list(row.values)[1])
    otu_to_genus[otu_id] = extract_genus(tax_str)

# 3. Aggregate OTU table to genus level
print("Aggregating to genus level...")
sample_cols = [c for c in otu.columns if c != 'OTU_Id' and c != 'OTU_ID' and not c.lower().startswith('tax')]
otu['genus'] = otu.iloc[:, 0].astype(str).map(otu_to_genus)
genus_abundance = otu.groupby('genus')[sample_cols].sum().T
genus_abundance.index.name = 'sample_id'

# Filter to genera present in ≥5 samples with ≥100 total reads
genus_total = genus_abundance.sum()
genus_presence = (genus_abundance > 0).sum()
genus_abundance = genus_abundance.loc[:, (genus_total >= 100) & (genus_presence >= 5)]

print(f"  Genus-level matrix: {genus_abundance.shape[0]:,} samples × {genus_abundance.shape[1]:,} genera")

# 4. Load contextual metadata and map sample IDs
print("Loading metadata...")
meta = pd.read_excel(DATA_DIR / 'AM_Contextual_Data_Master_Sheet-20180501.xlsx', engine='openpyxl')

# Convert numeric OTU column IDs to full Sample_IDs
# e.g., '12520' → '102.100.100/12520'
sample_id_map = {str(s): f'102.100.100/{s}' for s in sample_cols}

# 5. Find environment column in metadata
# Based on the metadata inspection earlier: 'broad land use' is the best environment classifier
env_col = 'broad land use'
if env_col in meta.columns:
    # Map sample IDs to environment categories
    sample_to_env = dict(zip(meta['Sample_ID'].astype(str), meta[env_col]))
    
    # Map numeric OTU IDs → Sample_ID → environment
    otu_to_env = {}
    for numeric_id in sample_cols:
        sample_id = sample_id_map.get(str(numeric_id), str(numeric_id))
        env = sample_to_env.get(sample_id, 'Unknown')
        otu_to_env[str(numeric_id)] = env
    
    # Aggregate genus abundance by environment
    genus_abundance['environment'] = genus_abundance.index.map(otu_to_env)
    genus_abundance = genus_abundance[genus_abundance['environment'] != 'Unknown']
    
    # Collapse rare environment categories (<10 samples)
    env_counts = genus_abundance['environment'].value_counts()
    valid_envs = env_counts[env_counts >= 10].index
    genus_abundance = genus_abundance[genus_abundance['environment'].isin(valid_envs)]
    
    print(f"\n  Environment categories: {list(valid_envs)}")
    print(f"  Samples per environment: {env_counts[valid_envs].to_dict()}")
    
    # 6. Compute Levins' B_std
    print("\nComputing Levins' B_std...")
    env_abundance = genus_abundance.groupby('environment').sum()
    env_props = env_abundance.div(env_abundance.sum(axis=0), axis=1)
    R = len(env_props.index)
    
    results = []
    for genus in env_props.columns:
        p = env_props[genus].values
        p = p[p > 0]
        if len(p) < 2:
            continue
        simpson = np.sum(p**2)
        if simpson == 0:
            continue
        B = 1 / simpson
        B_std = (B - 1) / (R - 1) if R > 1 else 0
        n_envs = len(p)
        results.append({
            'genus': genus,
            'levins_B_std': B_std,
            'n_envs_detected': n_envs,
            'total_environments': R
        })
    
    niche = pd.DataFrame(results)
    print(f"\n  Niche breadth computed for {len(niche):,} genera")
    print(f"  B_std range: {niche['levins_B_std'].min():.3f} – {niche['levins_B_std'].max():.3f}")
    print(f"  Mean B_std: {niche['levins_B_std'].mean():.3f}")
    
    # 7. Merge with 94-KO predictors from primary analysis
    print("\nMerging with 94-KO predictors...")
    trait_table = pd.read_csv('../data/genus_trait_table.csv')
    niche['genus_lower'] = niche['genus'].str.lower()
    
    merged = niche.merge(
        trait_table[['genus_lower', 'mean_n_metal_types', 'mean_n_defense_clusters', 
                      'mean_n_homeostasis_clusters', 'mean_metal_core_fraction']],
        on='genus_lower', how='inner'
    )
    
    print(f"  Overlap with primary analysis genera: {len(merged):,}")
    
    # 8. Save for PGLS
    merged.to_csv(DATA_DIR / 'aus_levinsB_merged.csv', index=False)
    print(f"\n✅ Saved: aus_levinsB_merged.csv ({len(merged):,} genera)")
    print(f"Ready for PGLS replication with Levins' B_std")
else:
    print(f"\n❌ Environment column '{env_col}' not found in metadata")
    print(f"Available columns: {meta.columns.tolist()}")

Loading OTU table...


  91,929 OTUs × 1,024 columns
Loading taxonomy...


Aggregating to genus level...


  Genus-level matrix: 1,023 samples × 1 genera
Loading metadata...



  Environment categories: ['Conservation and natural environments', 'Production from Dryland Agriculture and Plantations', 'Production from Relatively Natural Environments']
  Samples per environment: {'Conservation and natural environments': 818, 'Production from Dryland Agriculture and Plantations': 160, 'Production from Relatively Natural Environments': 33}

Computing Levins' B_std...

  Niche breadth computed for 1 genera
  B_std range: 0.238 – 0.238
  Mean B_std: 0.238

Merging with 94-KO predictors...
  Overlap with primary analysis genera: 0

✅ Saved: aus_levinsB_merged.csv (0 genera)
Ready for PGLS replication with Levins' B_std


In [42]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data/aus_microbiome')

# 1. Check metadata environment columns with actual data
meta = pd.read_excel(DATA_DIR / 'AM_Contextual_Data_Master_Sheet-20180501.xlsx', 
                     engine='openpyxl')

print("=== Environment-relevant columns with value counts ===")
env_cols = ['broad land use', 'General Ecological Zone', 'Vegetation Type',
            'Horizon', 'Depth', 'location description', 'Detailed land use']
for col in env_cols:
    if col in meta.columns:
        vc = meta[col].value_counts()
        print(f"\n--- {col} ({len(vc)} unique values) ---")
        print(vc.head(15).to_string())

# 2. Check sample ID matching
print("\n\n=== Sample ID matching ===")
# Generate mapped IDs
otu_sample_cols = [str(c) for c in range(12520, 13543)]  # 1023 columns
mapped_ids = [f'102.100.100/{s}' for s in otu_sample_cols]
meta_ids = set(meta['Sample_ID'].astype(str))

# Check overlap
overlap = set(mapped_ids) & meta_ids
print(f"Expected OTU samples: {len(otu_sample_cols)}")
print(f"Metadata samples: {len(meta_ids)}")
print(f"Overlap with mapping: {len(overlap)}")

if len(overlap) > 0:
    print(f"Overlap examples: {list(overlap)[:5]}")
else:
    # Check if meta IDs have a different prefix
    meta_prefixes = meta['Sample_ID'].str.extract(r'^([\d.]+)/')[0].value_counts()
    print(f"Metadata ID prefixes: {meta_prefixes.to_dict()}")
    
    # Check first few meta IDs
    print(f"First 10 meta Sample_IDs: {meta['Sample_ID'].head(10).tolist()}")

# 3. Fix genus extraction from taxonomy
print("\n\n=== Taxonomy genus extraction ===")
tax = pd.read_excel(DATA_DIR / 'BASE_16S_taxonomy.xlsx', engine='openpyxl')
print(f"Taxonomy columns: {tax.columns.tolist()}")
print(f"Sample genus values:\n{tax['genus'].value_counts().head(20)}")

# Extract clean genus names
def clean_genus(g):
    if pd.isna(g) or str(g).strip() == '' or 'unclassified' in str(g).lower():
        return None
    g = str(g).strip()
    if g.startswith('g__'):
        g = g[3:]
    return g if g else None

tax['genus_clean'] = tax['genus'].apply(clean_genus)
print(f"\nUsable genus assignments: {tax['genus_clean'].notna().sum()} / {len(tax)}")
print(f"Unique genera: {tax['genus_clean'].nunique()}")
print(f"Top 20 genera:\n{tax['genus_clean'].value_counts().head(20)}")

=== Environment-relevant columns with value counts ===

--- broad land use (5 unique values) ---
broad land use
Conservation and natural environments                    1123
Production from Dryland Agriculture and Plantations       174
Production from Relatively Natural Environments            34
Water                                                       6
Production from Irrigated Agriculture and Plantations       2

--- General Ecological Zone (12 unique values) ---
General Ecological Zone
Temperate         662
Polar             260
Arid              196
Coastal            90
Other (polar)      72
Tropical (wet)     57
Tropical (dry)     34
Montane            15
Mediterranian      13
Riverine            8
Other               8
Wet Tropics         2

--- Vegetation Type (13 unique values) ---
Vegetation Type
Woodland                386
Moss and lichen         332
Grassland               186
Other                   160
Shrubland               127
Forest                  126
Marsh/bog 

Taxonomy columns: ['OTUId', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
Sample genus values:
genus
unclassified                73776
g__Paenibacillus              790
g__Planctomyces               661
g__Aquicella                  598
g__Gemmata                    514
g__Bacillus                   394
g__Clostridium                351
g__Bdellovibrio               350
g__Candidatus_Solibacter      347
g__Flavisolibacter            315
g__DA101                      313
g__Nocardioides               291
g__Cystobacter                257
g__Asteroleplasma             250
g__Fimbriimonas               245
g__FFCH10602                  237
g__Pedobacter                 214
g__Candidatus_Koribacter      211
g__Alicyclobacillus           198
g__Kouleothrix                194
Name: count, dtype: int64

Usable genus assignments: 18152 / 91928
Unique genera: 933
Top 20 genera:
genus_clean
Paenibacillus            790
Planctomyces             661
Aquicella                

In [43]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data/aus_microbiome')

# ── 1. Load taxonomy and build OTU → genus map ────────────────────────
print("Loading taxonomy...")
tax = pd.read_excel(DATA_DIR / 'BASE_16S_taxonomy.xlsx', engine='openpyxl')

def clean_genus(g):
    if pd.isna(g) or str(g).strip() == '':
        return None
    g = str(g).strip()
    if 'unclassified' in g.lower():
        return None
    if g.startswith('g__'):
        g = g[3:]
    return g if g else None

tax['genus_clean'] = tax['genus'].apply(clean_genus)
otu_to_genus = dict(zip(tax['OTUId'].astype(str), tax['genus_clean']))
otu_to_genus = {k: v for k, v in otu_to_genus.items() if v is not None}

# ── 2. Load OTU table and aggregate to genus level ───────────────────
print("Loading OTU table and aggregating to genus level...")
otu = pd.read_csv(DATA_DIR / 'BASE_16S_OTU.csv.gz', compression='gzip')
otu['genus'] = otu['OTU_Id'].astype(str).map(otu_to_genus)
otu = otu.dropna(subset=['genus'])

sample_cols = [c for c in otu.columns if c not in ['OTU_Id', 'genus']]
genus_abundance = otu.groupby('genus')[sample_cols].sum().T

print(f"  Genus matrix: {genus_abundance.shape[0]} samples × {genus_abundance.shape[1]} genera")

# ── 3. Map sample IDs to environment categories ──────────────────────
print("Loading metadata...")
meta = pd.read_excel(DATA_DIR / 'AM_Contextual_Data_Master_Sheet-20180501.xlsx', engine='openpyxl')

# Numeric OTU column → full Sample ID: 12520 → 102.100.100/12520
def numeric_to_sample_id(numeric_id):
    return f'102.100.100/{numeric_id}'

# Build environment lookup: Sample_ID → General Ecological Zone
sample_to_env = dict(zip(meta['Sample_ID'].astype(str), meta['General Ecological Zone']))

# Map OTU column IDs to environment categories
numeric_to_env = {}
matched = 0
for col in sample_cols:
    sample_id = numeric_to_sample_id(col)
    env = sample_to_env.get(sample_id)
    if env is not None and pd.notna(env):
        numeric_to_env[col] = env
        matched += 1

print(f"  Sample-to-environment matches: {matched}/{len(sample_cols)}")

# Add environment column to genus abundance
genus_abundance['environment'] = genus_abundance.index.map(numeric_to_env)
genus_abundance = genus_abundance.dropna(subset=['environment'])

# Filter to environments with ≥10 samples
env_counts = genus_abundance['environment'].value_counts()
valid_envs = env_counts[env_counts >= 10].index
genus_abundance = genus_abundance[genus_abundance['environment'].isin(valid_envs)]

print(f"  Environment categories: {len(valid_envs)}")
print(f"  Samples per environment: {env_counts[valid_envs].to_dict()}")

# ── 4. Compute Levins' B_std ────────────────────────────────────────
print("\nComputing Levins' B_std...")
env_abundance = genus_abundance.groupby('environment').sum()
env_props = env_abundance.div(env_abundance.sum(axis=0), axis=1)
R = len(env_props.index)

results = []
for genus in env_props.columns:
    p = env_props[genus].values
    p = p[p > 0]
    if len(p) < 2:
        continue
    simpson = np.sum(p**2)
    if simpson == 0:
        continue
    B = 1 / simpson
    B_std = (B - 1) / (R - 1) if R > 1 else 0
    results.append({
        'genus': genus,
        'levins_B_std': B_std,
        'n_envs_detected': len(p),
        'total_environments': R
    })

niche = pd.DataFrame(results)
print(f"  Computed for {len(niche)} genera")
print(f"  B_std range: {niche['levins_B_std'].min():.3f} – {niche['levins_B_std'].max():.3f}")
print(f"  Mean B_std: {niche['levins_B_std'].mean():.3f}")

# ── 5. Merge with 94‑KO predictors from primary analysis ───────────
print("\nMerging with 94-KO predictors...")
trait_table = pd.read_csv('../data/genus_trait_table.csv')
niche['genus_lower'] = niche['genus'].str.lower()

merged = niche.merge(
    trait_table[['genus_lower', 'mean_n_metal_types', 'mean_n_defense_clusters', 
                  'mean_n_homeostasis_clusters', 'mean_metal_core_fraction']],
    on='genus_lower', how='inner'
)

print(f"  Overlap with primary analysis: {len(merged)} genera")

# ── 6. Prepare for PGLS and save ───────────────────────────────────
if len(merged) > 0:
    # Merge with genome size data for per‑Mb normalization
    genome_size = pd.read_csv('../data/genus_genome_size_gtdb.csv')
    genome_size['genus_lower'] = genome_size['genus_lower'].str.lower()
    merged = merged.merge(genome_size[['genus_lower', 'mean_genome_size_bp']], 
                          on='genus_lower', how='inner')
    merged['genome_size_mb'] = merged['mean_genome_size_bp'] / 1e6
    
    # Compute per‑Mb metrics
    for col in ['mean_n_metal_types', 'mean_n_defense_clusters', 'mean_n_homeostasis_clusters']:
        merged[f'{col}_per_mb'] = merged[col] / merged['genome_size_mb']
    
    # Z‑score predictors
    from sklearn.preprocessing import StandardScaler
    for col in ['mean_n_metal_types_per_mb', 'mean_n_defense_clusters_per_mb', 
                'mean_n_homeostasis_clusters_per_mb']:
        vals = merged[col].values.reshape(-1, 1)
        merged[f'{col}_z'] = StandardScaler().fit_transform(vals).flatten()
    
    # Save
    pgls_input = merged[['genus_lower', 'levins_B_std',
                         'mean_n_metal_types_per_mb_z', 
                         'mean_n_defense_clusters_per_mb_z',
                         'mean_n_homeostasis_clusters_per_mb_z']].dropna()
    
    out_path = DATA_DIR / 'aus_levinsB_pgls_input.csv'
    pgls_input.to_csv(out_path, index=False)
    
    print(f"\n✅ PGLS input saved: {out_path}")
    print(f"   {len(pgls_input)} genera ready for replication PGLS")
    print(f"\nNext: Rscript scripts/pgls_mgnify_validation.R \\")
    print(f"       {out_path} data/gtdb_bac_genus_pruned.tree \\")
    print(f"       data/aus_replication_pgls.csv")
else:
    print("\n❌ No overlapping genera — replication not possible with this dataset")

Loading taxonomy...


Loading OTU table and aggregating to genus level...


  Genus matrix: 1023 samples × 933 genera
Loading metadata...


  Sample-to-environment matches: 1003/1023
  Environment categories: 8
  Samples per environment: {'Temperate': 429, 'Polar': 258, 'Arid': 152, 'Tropical (wet)': 57, 'Coastal': 39, 'Tropical (dry)': 22, 'Montane': 15, 'Mediterranian': 13}

Computing Levins' B_std...
  Computed for 812 genera
  B_std range: 0.000 – 0.470
  Mean B_std: 0.131

Merging with 94-KO predictors...
  Overlap with primary analysis: 638 genera



✅ PGLS input saved: ../data/aus_microbiome/aus_levinsB_pgls_input.csv
   482 genera ready for replication PGLS

Next: Rscript scripts/pgls_mgnify_validation.R \
       ../data/aus_microbiome/aus_levinsB_pgls_input.csv data/gtdb_bac_genus_pruned.tree \
       data/aus_replication_pgls.csv


In [46]:
!/home/hmacgregor/r_env/bin/Rscript ../scripts/pgls_mgnify_validation.R \
       ../data/aus_microbiome/aus_levinsB_pgls_input.csv ../data/gtdb_bac_genus_pruned.tree \
       ../data/aus_replication_pgls.csv


=== MGnify MAG validation PGLS ===
Input:  ../data/aus_microbiome/aus_levinsB_pgls_input.csv
Tree:   ../data/gtdb_bac_genus_pruned.tree
Output: ../data/aus_replication_pgls.csv

Loaded 482 genera from input CSV
Tree has 2283 tips
After tree pruning: 482 genera



[PGLS] biome_H_std ~ ko_per_mb_total_z  (n=482)
  SKIP: predictor has zero or near-zero variance

[PGLS] biome_H_std ~ ko_per_mb_tier1_z  (n=482)
  SKIP: predictor has zero or near-zero variance

[PGLS] biome_H_std ~ ko_per_mb_tier2_z  (n=482)
  SKIP: predictor has zero or near-zero variance

Tip: tree_sub$edge.length summary:
    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
0.006582 0.050666 0.146116 0.190958 0.253931 1.323838 
Error: No models converged.
Execution halted


In [48]:
# Check the actual column names in the saved CSV
!head -1 ../data/aus_microbiome/aus_levinsB_pgls_input.csv

genus_lower,levins_B_std,mean_n_metal_types_per_mb_z,mean_n_defense_clusters_per_mb_z,mean_n_homeostasis_clusters_per_mb_z


In [49]:
import pandas as pd
df = pd.read_csv('../data/aus_microbiome/aus_levinsB_pgls_input.csv')
df = df.rename(columns={
    'levins_B_std': 'biome_H_std',
    'mean_n_metal_types_per_mb_z': 'ko_per_mb_total_z',
    'mean_n_defense_clusters_per_mb_z': 'ko_per_mb_tier1_z',
    'mean_n_homeostasis_clusters_per_mb_z': 'ko_per_mb_tier2_z'
})
df[['genus_lower','biome_H_std','ko_per_mb_total_z','ko_per_mb_tier1_z','ko_per_mb_tier2_z']].to_csv(
    '../data/aus_microbiome/aus_levinsB_pgls_input_fixed.csv', index=False)
print(f'Saved {len(df)} rows')

Saved 482 rows


In [50]:
!/home/hmacgregor/r_env/bin/Rscript ../scripts/pgls_mgnify_validation.R \
    ../data/aus_microbiome/aus_levinsB_pgls_input_fixed.csv \
    ../data/gtdb_bac_genus_pruned.tree \
    ../data/aus_replication_pgls.csv


=== MGnify MAG validation PGLS ===
Input:  ../data/aus_microbiome/aus_levinsB_pgls_input_fixed.csv
Tree:   ../data/gtdb_bac_genus_pruned.tree
Output: ../data/aus_replication_pgls.csv

Loaded 482 genera from input CSV
Tree has 2283 tips
After tree pruning: 482 genera



[PGLS] biome_H_std ~ ko_per_mb_total_z  (n=482)


  lambda=0.2670  beta=-0.0023  SE=0.0054  t=-0.430  p=0.6674  deltaAIC=1.82

[PGLS] biome_H_std ~ ko_per_mb_tier1_z  (n=482)


  lambda=0.2947  beta=0.0067  SE=0.0058  t=1.152  p=0.2499  deltaAIC=0.69

[PGLS] biome_H_std ~ ko_per_mb_tier2_z  (n=482)


  lambda=0.2749  beta=0.0041  SE=0.0048  t=0.847  p=0.3972  deltaAIC=1.28

Saved: ../data/aus_replication_pgls.csv  (3 models)

Summary:
               predictor n_taxa    lambda         beta   p_value delta_AIC
Value  ko_per_mb_total_z    482 0.2669773 -0.002311352 0.6674035 1.8214624
Value1 ko_per_mb_tier1_z    482 0.2946605  0.006688471 0.2498759 0.6923163
Value2 ko_per_mb_tier2_z    482 0.2748662  0.004062198 0.3972135 1.2797906
